In [1]:
import json
import os
import time
import base64
import pandas as pd
import requests 
os.environ['http_proxy'] = 'http://localhost:7890'
os.environ['https_proxy'] = 'http://localhost:7890'
api_key ='sk-m6qcnmUk5ZgRjZfK9cEeAf868972412fBa85B3F1187d4b0f'

In [2]:
with open('revised_vary_element_qa_pairs.json', 'r') as file:
    vary_element_qa_pairs = json.load(file)
with open('revised_vary_element_annotations.json', 'r') as file:
    vary_element_annotations = json.load(file)

In [3]:
def gpt_cost(number):
    each_cost = number / 1000 * 0.01 * 7.2
    return round(each_cost, 4)

def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}


def get_gpt_4v_reply(image_url, final_question):
    base64_image = encode_image(image_url)
    payload = {
      "model": "gpt-4o-2024-05-13",
      "messages": [
        {
          "role": "user",
          "content": [
            {
              "type": "text",
              "text": final_question
            },
            {
              "type": "image_url",
              "image_url": {
                "url": f"data:image/png;base64,{base64_image}",
#                 'detail':'high'
              },
            }
          ]
        }
      ],
      "max_tokens": 1000
    }

    response = requests.post("https://svip.yi-zhan.top/v1/chat/completions", headers=headers, json=payload)
#     gpt_4v_total_tokens = response.json()['usage']['total_tokens']
    print(response.json())
    gpt_4v_reply = response.json()['choices'][0]['message']
    
    return gpt_4v_reply

#     try:
#         response = requests.post("https://vip.yi-zhan.top/v1/chat/completions", headers=headers, json=payload)
#         print(response.status_code)
#         response.raise_for_status()  # 这将抛出HTTPError，如果响应状态码不是2xx

#         # 解析响应内容
#         gpt_4v_reply = response.json()
#         print(gpt_4v_reply)
#         return gpt_4v_reply

#     except requests.exceptions.HTTPError as http_err:
#         # 处理HTTP错误
#         return f"HTTP请求错误: {http_err}"
#     except requests.exceptions.ConnectionError as conn_err:
#         # 处理连接错误
#         return f"连接错误: {conn_err}"
#     except requests.exceptions.Timeout as timeout_err:
#         # 处理请求超时
#         return f"请求超时: {timeout_err}"
#     except requests.exceptions.RequestException as req_err:
#         # 处理其他请求相关错误
#         return f"请求错误: {req_err}"
#     except Exception as e:
#         # 处理其他未知错误
#         return f"未知错误: {e}"


## 主体代码

In [ ]:
gpt4_turbo_total_cost = 0
gpt4v_total_cost = 0
start_length = 0
total_questions = 0
gap = False
annotation_id = 0
total_question_limit = 4020
for _, each_annotation in enumerate(vary_element_annotations[annotation_id:]):
    total_charts = []
    if gap == True:
        break
    # print(each_annotation)
    begin = time.time()
    image_path = each_annotation['changed_image']
    vary_element = each_annotation['vary_element']
    vary_type = each_annotation['vary_type']
    # print(image_index)
    image_url = f'vary_element/{vary_element}/{vary_type}/'  + image_path
    image_type = each_annotation['type']
    index = each_annotation['id']
    for i, each_qa_pair in enumerate(vary_element_qa_pairs[start_length:]):
        each_chart = {}
        each_chart['image_url'] = image_url
        each_chart['image_type'] = image_type
        task_category = each_qa_pair['type']
        each_chart['task_category'] = task_category
        each_chart['vary_element'] = each_qa_pair['vary_element']
        each_chart['vary_type'] = each_qa_pair['vary_type']

        if index == each_qa_pair['image_index'] and vary_element == each_qa_pair['vary_element'] and vary_type == each_qa_pair['vary_type']:
            start_length += 1

            if task_category == 'reasoning' and ('variance' in each_qa_pair['QA_pairs'][0]['fill_the_blank'][0] or 'deviation' in each_qa_pair['QA_pairs'][0]['fill_the_blank'][0]):
                continue
            total_questions += 4

            print(f"{index}, {task_category}, {vary_type}")
            for each_question_format in each_qa_pair['QA_pairs']:
                each_key = list(each_question_format.keys())[0]

                final_question = each_question_format[each_key][0]
                annotation = each_question_format[each_key][1]

                qwen_reply = get_gpt_4v_reply(image_url, final_question)

                each_chart[each_key + ' question'] = final_question
                each_chart[each_key + ' annotation'] = annotation
                each_chart[each_key + ' gpt4o'] = qwen_reply
            each_chart['start_length'] = start_length
            each_chart['annotation_id'] = annotation_id
            each_chart['pair_index'] = each_qa_pair['pair_index']
            total_charts.append(each_chart)
        else:
            break

    annotation_id += 1
    with open(f'results/gpt4o/gpt4o_element_{annotation_id}_{start_length}.json', 'w') as file:
        json.dump(total_charts, file, indent=4)
    print(f"已保存{annotation_id}/{len(vary_element_annotations)}张图表-{start_length}/{len(vary_element_qa_pairs)}个问题的结果。")



1098, data retrieval, original
